In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
)
from imblearn.over_sampling import SMOTE

# ── Configuración global ──────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.30

RUTAS = {
    "matrimonios": "data_matrimonios",
    "divorcios":   "data_divorcios",
}

ARCHIVOS = {
    "matrimonios": [
        "matrimonios_2019.sav", "matrimonios_2020.sav", "matrimonios_2021.sav",
        "matrimonios_2022.sav", "matrimonios_2023.sav", "matrimonios_extra.sav",
    ],
    "divorcios": ["divorcios_parte1.sav", "divorcios_parte2.sav"],
}

VALORES_INVALIDOS = {98, 99, 998, 999, 9998, 9999}

RF_CONFIGS = [
    {"n_estimators": 50,  "max_depth": 5,  "label": "Modelo 1"},
    {"n_estimators": 100, "max_depth": 10, "label": "Modelo 2"},
    {
        "n_estimators": 200, "max_depth": 15,
        "min_samples_split": 5, "class_weight": "balanced",
        "label": "Modelo 3",
    },
]

print("Librerías cargadas y configuración lista.")

In [ ]:
def cargar_archivos(tipo: str, label: int) -> pd.DataFrame:
    dfs = []
    for archivo in ARCHIVOS[tipo]:
        ruta = os.path.join(RUTAS[tipo], archivo)
        df = pd.read_spss(ruta)
        df["TIPO_REGISTRO"] = tipo.upper()
        df["DIVORCIO"] = label
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def limpiar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.upper().str.strip().str.replace(" ", "_", regex=False)
    return df


def reemplazar_invalidos(df: pd.DataFrame) -> pd.DataFrame:
    invalidos_str = {str(v) for v in VALORES_INVALIDOS}
    invalidos_lista = list(VALORES_INVALIDOS | invalidos_str)

    cols_cat = df.select_dtypes(include="category").columns
    df[cols_cat] = df[cols_cat].astype(object)

    return df.replace(invalidos_lista, np.nan)


def limpiar_edades(df: pd.DataFrame) -> pd.DataFrame:
    cols_edad = [c for c in df.columns if "EDAD" in c]
    for col in cols_edad:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[~df[col].between(14, 100), col] = np.nan
    return df


def crear_variables_derivadas(df: pd.DataFrame) -> pd.DataFrame:
    col_hombre = next(
        (c for c in df.columns if "EDAD" in c and any(s in c for s in ("HOM", "HOMBRE"))), None
    )
    col_mujer = next(
        (c for c in df.columns if "EDAD" in c and any(s in c for s in ("MUJ", "MUJER"))), None
    )

    if col_hombre and col_mujer:
        df["DIFERENCIA_EDAD"] = (df[col_hombre] - df[col_mujer]).abs()
        df["EDAD_PROMEDIO"] = (df[col_hombre] + df[col_mujer]) / 2
        print(f"Variables derivadas creadas desde '{col_hombre}' y '{col_mujer}'")
    else:
        print("No se encontraron columnas de edad por sexo.")

    return df


def crear_target(df: pd.DataFrame) -> pd.DataFrame:
    col_num = next(
        (c for c in df.columns if "NUM" in c and "MATRIMONIO" in c), None
    )

    if col_num:
        df["SEGUNDO_MATRIMONIO"] = (
            pd.to_numeric(df[col_num], errors="coerce").ge(2).astype(int)
        )
        print(f"Target creado desde: '{col_num}'")
    else:
        print("No se detectó columna de número de matrimonio. SEGUNDO_MATRIMONIO = 0.")
        df["SEGUNDO_MATRIMONIO"] = 0

    return df


# Carga
df_matrimonios = cargar_archivos("matrimonios", label=0)
df_divorcios = cargar_archivos("divorcios", label=1)

print(f"Matrimonios: {df_matrimonios.shape} | Divorcios: {df_divorcios.shape}")

# Pipeline de limpieza
df_master = (
    pd.concat([df_matrimonios, df_divorcios], ignore_index=True, sort=False)
    .pipe(limpiar_columnas)
    .pipe(reemplazar_invalidos)
    .pipe(limpiar_edades)
    .pipe(crear_variables_derivadas)
    .pipe(crear_target)
)

print(f"\nDataset inicial: {df_master.shape}")


# =========================
# CREAR SEGUNDO MATRIMONIO SINTÉTICO
# =========================

df_matrimonios_base = df_master[df_master["DIVORCIO"] == 0].copy()

df_sample = df_matrimonios_base.sample(frac=0.10, random_state=42)
df_segundo = df_sample.copy()

if "EDADHOM" in df_segundo.columns:
    df_segundo["EDADHOM"] = df_segundo["EDADHOM"] + np.random.randint(3, 10, size=len(df_segundo))

if "EDADMUJ" in df_segundo.columns:
    df_segundo["EDADMUJ"] = df_segundo["EDADMUJ"] + np.random.randint(3, 10, size=len(df_segundo))

if "EDADHOM" in df_segundo.columns and "EDADMUJ" in df_segundo.columns:
    df_segundo["DIFERENCIA_EDAD"] = abs(df_segundo["EDADHOM"] - df_segundo["EDADMUJ"])
    df_segundo["EDAD_PROMEDIO"] = (df_segundo["EDADHOM"] + df_segundo["EDADMUJ"]) / 2

df_master["SEGUNDO_MATRIMONIO"] = 0
df_segundo["SEGUNDO_MATRIMONIO"] = 1

df_master = pd.concat([df_master, df_segundo], ignore_index=True)

print("\nDataset final:", df_master.shape)

print("\nBalance final:")
print(df_master["SEGUNDO_MATRIMONIO"].value_counts())

print("\nBalance final (%):")
print(df_master["SEGUNDO_MATRIMONIO"].value_counts(normalize=True).mul(100).round(2))

print("\n Funciones de preprocesamiento y dataset final definidos.")

In [ ]:
# Paso 3: Verificación del dataset final

print("Dataset final:", df_master.shape)

print("\nBalance SEGUNDO_MATRIMONIO:")
print(df_master["SEGUNDO_MATRIMONIO"].value_counts())

print("\nBalance SEGUNDO_MATRIMONIO (%):")
print(df_master["SEGUNDO_MATRIMONIO"].value_counts(normalize=True).mul(100).round(2))

print("\nPrimeras filas del dataset:")
display(df_master.head())

In [ ]:
# Features numéricas (excluye target)
cols_modelo = [
    c for c in df_master.select_dtypes(include="number").columns
    if c != "SEGUNDO_MATRIMONIO"
]
X = df_master[cols_modelo].fillna(df_master[cols_modelo].median())
y = df_master["SEGUNDO_MATRIMONIO"]

print(f"Features seleccionadas ({len(cols_modelo)}): {cols_modelo}")

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Escalado (fit solo en train)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=cols_modelo)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=cols_modelo)

# SMOTE (sobre datos ya escalados)
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

X_train_final = pd.DataFrame(X_train_smote, columns=cols_modelo)
y_train_final = pd.Series(y_train_smote, name="SEGUNDO_MATRIMONIO")
X_test_final  = X_test_scaled.reset_index(drop=True)
y_test_final  = y_test.reset_index(drop=True)

print(f"\nTrain (post-SMOTE): {X_train_final.shape}")
print(f"Test:               {X_test_final.shape}")
print("\nBalance train post-SMOTE:")
print(pd.Series(y_train_smote).value_counts())

In [ ]:
X_train_final.to_csv("X_train_final.csv", index=False)
y_train_final.to_csv("y_train_final.csv", index=False)
X_test_final.to_csv("X_test_final.csv",   index=False)
y_test_final.to_csv("y_test_final.csv",   index=False)
df_master.to_csv("df_master_limpio.csv",  index=False)

print(" Datasets guardados: X_train_final, y_train_final, X_test_final, y_test_final, df_master_limpio")